# MMF Sample Data Generator

Generates synthetic time series in MMF format (`unique_id`, `ds`, `y`) and writes to `{catalog}.{schema}.{use_case}_train_data`.

**Pattern mix:** 40% seasonal · 20% trending · 20% mixed (trend + season) · 10% intermittent · 10% noisy.

In [ ]:
catalog        = "{catalog}"
schema         = "{schema}"
use_case       = "{use_case}"
n_series       = {n_series}
freq           = "{freq}"
history_length = {history_length}

In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql import functions as F

np.random.seed(42)

# Build date range based on frequency
if freq == "D":
    pd_freq = "D"
elif freq == "W":
    pd_freq = "W-SUN"
elif freq == "M":
    pd_freq = "ME"
else:
    pd_freq = "h"

dates = pd.date_range(end="2024-12-31", periods=history_length, freq=pd_freq)

# Seasonal period per frequency
if freq == "H":
    period = 24
elif freq == "D":
    period = 7
elif freq == "W":
    period = 52
else:
    period = 12

t = np.arange(history_length)

# Pattern distribution
n_seasonal     = max(1, int(n_series * 0.40))
n_trending     = max(1, int(n_series * 0.20))
n_mixed        = max(1, int(n_series * 0.20))
n_intermittent = max(1, int(n_series * 0.10))
n_noisy        = max(0, n_series - n_seasonal - n_trending - n_mixed - n_intermittent)

records = []

def add_series(uid, y_arr):
    for j, d in enumerate(dates):
        records.append([uid, d, round(float(max(0.0, y_arr[j])), 4)])

# Seasonal (40%): clear periodicity
for i in range(n_seasonal):
    base  = np.random.uniform(50, 500)
    amp   = np.random.uniform(0.2, 0.5) * base
    phase = np.random.uniform(0, 2 * np.pi)
    noise = np.random.uniform(0.05, 0.15) * base
    y = base + amp * np.sin(2 * np.pi * t / period + phase) + np.random.normal(0, noise, history_length)
    add_series("seasonal_{:03d}".format(i + 1), y)

# Trending (20%): linear growth
for i in range(n_trending):
    start = np.random.uniform(10, 200)
    slope = np.random.uniform(0.001, 0.005) * start
    noise = np.random.uniform(0.05, 0.10) * start
    y = start + slope * t + np.random.normal(0, noise, history_length)
    add_series("trending_{:03d}".format(i + 1), y)

# Mixed (20%): trend + seasonality
for i in range(n_mixed):
    base  = np.random.uniform(50, 300)
    slope = np.random.uniform(0.001, 0.003) * base
    amp   = np.random.uniform(0.15, 0.35) * base
    phase = np.random.uniform(0, 2 * np.pi)
    noise = np.random.uniform(0.05, 0.10) * base
    y = base + slope * t + amp * np.sin(2 * np.pi * t / period + phase) + np.random.normal(0, noise, history_length)
    add_series("mixed_{:03d}".format(i + 1), y)

# Intermittent (10%): sparse demand with zeros
for i in range(n_intermittent):
    demand_prob = np.random.uniform(0.1, 0.4)
    mean_demand = np.random.uniform(5, 50)
    y = np.where(
        np.random.binomial(1, demand_prob, history_length),
        np.random.exponential(mean_demand, history_length),
        0.0,
    )
    add_series("intermittent_{:03d}".format(i + 1), y)

# Noisy (remaining): high variance, low signal
for i in range(n_noisy):
    base  = np.random.uniform(20, 200)
    noise = np.random.uniform(0.3, 0.6) * base
    y = np.abs(base + np.random.normal(0, noise, history_length))
    add_series("noisy_{:03d}".format(i + 1), y)

df = pd.DataFrame(records, columns=["unique_id", "ds", "y"])
print("Generated {:,} rows for {} series".format(len(df), df["unique_id"].nunique()))

In [ ]:
sdf = spark.createDataFrame(df)

if freq == "H":
    sdf = sdf.withColumn("ds", F.col("ds").cast("timestamp"))
else:
    sdf = sdf.withColumn("ds", F.col("ds").cast("date"))

output_table = catalog + "." + schema + "." + use_case + "_train_data"
(
    sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

total      = sdf.count()
n_distinct = sdf.select("unique_id").distinct().count()
row = sdf.agg(F.min("ds").alias("mn"), F.max("ds").alias("mx")).collect()[0]
print("Written to:", output_table)
print("Series    :", n_distinct)
print("Rows      :", "{:,}".format(total))
print("Range     :", row["mn"], "to", row["mx"])